In [2]:
import streamlit as st

In [3]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!python.exe -m pip install --upgrade pip

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 5.9 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 4.0 MB/s eta 0:00:01
   ------------------------------------- -- 1.8/2.0 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 3.2 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.1
    Uninstalling pip-26.2.1:
      Successfully uninstalled pip-26.2.1


In [5]:
print("Hello World!")

Hello World!


In [9]:
st.title("Retail Sales Dashboard!")
st.header("Sales Overview")
st.write("Completed and Shipped Orders")
st.metric("Orders", 11439)
# st.dataframe()

2026-11-05 16:13:34.233 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 16:13:34.234 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 16:13:34.236 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 16:13:34.237 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 16:13:34.238 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 16:13:34.240 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 16:13:34.241 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 16:13:34.244 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [36]:
def load_data():
    customers = pd.read_csv("data/customers.csv")
    products = pd.read_csv("data/products.csv")
    orders = pd.read_csv("data/orders.csv")
    items = pd.read_csv("data/order_items.csv")
    return customers, products, orders, items

customers, products, orders, items = load_data()

In [54]:
# ============================================================
# 2. HELPER FUNCTION FOR NUMERIC CLEANING
# ============================================================

def clean_money(series):
    """
    Convert values such as:

        1,250.50
        ₱1,250.50
        1250.50

    into numeric values.

    Invalid values become NaN.
    """

    return pd.to_numeric(
        series.astype("string")
        .str.replace("₱", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce",
    )

In [51]:
# ----------------------------------------------------
# B. REMOVE EXACT DUPLICATES
# ----------------------------------------------------

customers = (
    customers
    .drop_duplicates()
    .copy()
)

products = (
    products
    .drop_duplicates()
    .copy()
)

orders = (
    orders
    .drop_duplicates()
    .copy()
)

items = (
    items
    .drop_duplicates()
    .copy()
)

In [43]:
category_map = {
    "electronics": "Electronics",
    "elec": "Electronics",
    "home & kitchen": "Home & Kitchen",
    "home and kitchen": "Home & Kitchen",
    "books": "Books",
    "book": "Books",
    "groceries": "Groceries",
    "grocery": "Groceries",
    "toys": "Toys",
    "toy": "Toys",
    "beauty": "Beauty",
    "beauty & care": "Beauty",
    "fashion": "Fashion",
    "fashon": "Fashion",
    "sports": "Sports",
    "sports & outdoor": "Sports"
}

products["category_clean"] = (
    products["category"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(category_map)
    .fillna("Unknown")
)

In [15]:
products

,product_id,product_name,brand,category,unit_cost,unit_price,stock_qty,uom,launch_date,category_clean
0,PROD00001,KidJoy Beauty 963,KidJoy,Beauty,44.17,64.5,125.0,pc,2022-01-18,beauty
1,PROD00002,HomeEase Home 368,HomeEase,Home and Kitchen,37.18,70.39,131.0,pcs,2022-11-25,Home & Kitchen
2,PROD00003,FreshCart Fashion 911,FreshCart,Fashion,46.29,₱70.26,41.0,piece,2023-09-04,fashion
3,PROD00004,NovaTech Toys 753,NovaTech,Toys,43.66,62.46,149.0,pcs,2023-08-12,Toys
4,PROD00005,GlowCo Books 916,GlowCo,Books,64.59,82.02,162.0,bottle,2024-05-12,Books
...,...,...,...,...,...,...,...,...,...,...
795,PROD00796,KidJoy Elec 294,KidJoy,Elec,83.80,132.12,111.0,bottle,2023-08-15,Electronics
796,PROD00797,SunPeak Books 182,SunPeak,Books,53.67,62.54,163.0,bottle,2023-01-05,Books
797,PROD00798,NovaTech Groceries 889,NovaTech,Groceries,63.54,112.46,155.0,set,2022-08-09,Groceries
798,PROD00799,Acme Fashion 991,Acme,Fashion,25.27,34.96,169.0,bottle,2024-06-19,fashion


In [55]:
# ----------------------------------------------------
# D. CLEAN ORDER ITEM VALUES
# ----------------------------------------------------

items["quantity"] = pd.to_numeric(
    items["quantity"],
    errors="coerce"
)

items["unit_price_clean"] = clean_money(
    items["unit_price_at_purchase"]
)

# Keep valid quantity and price rows only
items = items[
    (items["quantity"] > 0)
    &
    items["unit_price_clean"].notna()
].copy()

In [59]:
# --------------------------------------------------------
# E. CLEAN ORDERS
# --------------------------------------------------------

# Your dataset contains several date formats.
# format="mixed" allows Pandas to interpret them.

orders["order_date"] = pd.to_datetime(
    orders["order_datetime"],
    format="mixed",
    errors="coerce",
)

# Convert discount to numeric

orders["discount_pct"] = pd.to_numeric(
    orders["discount_pct"],
    errors="coerce",
)

# Missing discount means no discount

orders["discount_pct"] = (
    orders["discount_pct"]
    .fillna(0)
)

# Convert shipping fee

orders["shipping_fee"] = pd.to_numeric(
    orders["shipping_fee"],
    errors="coerce",
)

# Keep valid discount percentages only.
# Example:
# 0.10 = 10%
# 0.25 = 25%

orders = orders[
    orders["discount_pct"].between(0, 1)
].copy()

In [60]:
# --------------------------------------------------------
# F. REFERENTIAL INTEGRITY
# --------------------------------------------------------

# Remove orders whose customer_id does not exist
# in the customer master file.

orders = orders[
    orders["customer_id"].isin(
        customers["customer_id"]
    )
].copy()

# Remove order items whose product does not exist

items = items[
    items["product_id"].isin(
        products["product_id"]
    )
].copy()

# Remove order items whose order does not exist

items = items[
    items["order_id"].isin(
        orders["order_id"]
    )
].copy()

In [61]:
# --------------------------------------------------------
# G. BUILD ITEM-LEVEL ANALYTICAL DATAFRAME
# --------------------------------------------------------

sales = (
    items
    .merge(
        orders,
        on="order_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        products[
            [
                "product_id",
                "product_name",
                "brand",
                "category_clean",
            ]
        ],
        on="product_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        customers[
            [
                "customer_id",
                "region_name",
                "city",
                "loyalty_tier",
            ]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
)

In [62]:
# --------------------------------------------------------
# H. CREATE ANALYTICAL FIELDS
# --------------------------------------------------------

# Gross Sales
#
# Quantity × Unit Price

sales["gross_sales"] = (
    sales["quantity"]
    *
    sales["unit_price_clean"]
)

# Net Sales
#
# Gross Sales after discount

sales["net_sales"] = (
    sales["gross_sales"]
    *
    (
        1
        -
        sales["discount_pct"]
    )
)

# Create month column

sales["month"] = (
    sales["order_date"]
    .dt.to_period("M")
    .astype("string")
)

In [63]:
# --------------------------------------------------------
# I. KEEP ONLY FULFILLED ORDERS
# --------------------------------------------------------

fulfilled_sales = sales[
    sales["order_status"].isin(
        [
            "Completed",
            "Shipped",
        ]
    )
].copy()

# Remove rows with unparseable dates

fulfilled_sales = fulfilled_sales[
    fulfilled_sales["order_date"].notna()
].copy()

return fulfilled_sales

SyntaxError: 'return' outside function (1780581463.py, line 20)